# 保存した比較モデルで予測する

保存済みの Ridge 比較を一覧から選びます。追加の学習は行わず、保存された候補モデルで x1・x2・x3 から予測します。1回に1〜64行を指定できます。


In [ ]:
from IPython.display import JSON, display
from tools.local_experiment.mlflow.comparison_predict import predict, load_saved
from tools.local_experiment.mlflow.comparison_predict import ComparisonPredictionError
from tools.local_experiment.notebook_lab.comparison_saved import SavedComparisonError
from tools.local_experiment.mlflow.comparison_runs import collect, ComparisonRunsError
inventory = None
try:
    inventory = collect()
except Exception as error:
    print("比較一覧を確認できませんでした。", str(error) if isinstance(error, ComparisonRunsError) else "COMPARISON_RUNS_FAILED")
else:
    display(JSON(inventory))


一覧の runs に表示された comparison_token をコピーしてください。下の特徴量を編集して予測セルを実行できます。未完了や未検証の比較は選択できません。


In [ ]:
comparison_token = ""
request = {"schema_version": 1, "rows": [
    {"x1": 0.0, "x2": 0.0, "x3": 0.0},
    {"x1": 1.0, "x2": -0.5, "x3": 0.25},
]}


In [ ]:
prediction = None
if inventory is None or inventory["result"] != "complete":
    print("比較一覧を確認できませんでした。最初のセルの結果を確認してください。")
elif comparison_token not in {run["comparison_token"] for run in inventory["runs"]}:
    print("一覧の runs に表示された比較実行の token を入力してください。")
else:
    try:
        saved = load_saved(comparison_token)
        prediction = predict(comparison_token, request)
    except (SavedComparisonError, ComparisonPredictionError) as error:
        print("予測を完了できませんでした。token と入力を確認してください。", str(error))
    else:
        print("保存した候補モデルの alpha:", saved["comparison"]["candidate"]["alpha"])
        for row, value in zip(request["rows"], prediction["predictions"]):
            print(row, "→", value)
        display(JSON(prediction))


この予測は保存データで学習した候補の計算結果です。未知のデータでの精度を保証しません。Notebook を保存すると、入力と予測結果を再読込できます。新しい MLflow run・Reader・LLM 呼出しはありません。
